In [ ]:
import os

# Set your directory
os.chdir(r"C:\Users\aldoh\OneDrive\Escritorio\Agent x Agentx2")
print(f"Current directory: {os.getcwd()}")

In [ ]:

#!python preprocess_metrics.py
#!python preprocess_sdm.py
#!python preprocess_publications.py
#!python merge_all.py

In [1]:
# Installed Dependencies
#%pip install crewai
#%pip install python-dotenv
#%pip install anthropic


import os
from dotenv import load_dotenv
import anthropic
from crewai import Agent, Task, Crew, Process, LLM
import re
from collections import defaultdict
import json
from typing import List, Dict, Any 
from datetime import datetime

load_dotenv()

# ============================================================================
# LLM CONFIGURATIONS -------- Anthropic and OpenAI
# ============================================================================
#########################ANTHROPIC#################################
llm = LLM(
    model="claude-sonnet-4-5-20250929",
    api_key=os.getenv("ANTHROPIC_API_KEY"),
)
llmc = LLM(                                 ######Cheaper
    model="claude-3-5-haiku-20241022",    
    api_key=os.getenv("ANTHROPIC_API_KEY"),
)
#########################OPENAI####################################
llmgpt = LLM(
    model="gpt-4o-2024-08-06",        
    api_key=os.getenv("OPENAI_API_KEY"),
)
llmgptc = LLM(                              ######Cheaper    
    model="gpt-4o-mini-2024-07-18",   
    api_key=os.getenv("OPENAI_API_KEY"),
    response_format={"type": "json_object"},             
)
# ============================================================================
# IMPORT SCHEMAS
# ============================================================================

from schemas import CampusInsight

# ============================================================================
# LOAD UNIFIED DATA
# ============================================================================
print("📖 Loading unified campus data...")

with open('unified_campus_data.json', 'r', encoding='utf-8') as f:
    unified_data = json.load(f)

campuses = unified_data['campus_insights']
print(f"✅ Loaded {len(campuses)} campuses\n")

# ============================================================================
# AGENT DEFINITION
# ============================================================================
insight_agent = Agent(
    role="Analista de Performance Digital",
    goal="Generar insights profundos en español sobre el desempeño de redes sociales de cada campus, analizando métricas, contenido y scores de salud de marca",
    backstory="""Eres un experto analista de social media con años de experiencia en educación superior.
    Sabes interpretar métricas de engagement, identificar tendencias en contenido, y evaluar la salud de marca.
    Generas insights claros, concisos y accionables en español profesional.
    Analizas tanto números (cambios porcentuales, scores) como contenido (temas, emociones, narrativas).""",
    llm=llm,
    verbose=False,
    max_iter=3,
    allow_delegation=False,
)

# ============================================================================
# TASK DEFINITION
# ============================================================================
insight_task = Task(
    description="""Analiza los datos completos del campus y genera un insight en español siguiendo este formato EXACTO:

**DATOS DEL CAMPUS:**
{campus_data}

**MES DE ANÁLISIS:** Septiembre 2025

**FORMATO REQUERIDO:**
Escribe UN SOLO párrafo en español de EXACTAMENTE 95-100 palabras que incluya:

1. **Apertura**: "En septiembre 2025, el Campus [nombre] mostró un desempeño [categoría]" 
   - Usa SOLO la categoría (deficiente/regular/satisfactorio/sobresaliente/excepcional) del campo totales.salud_de_marca_categoria
   - NO menciones números de puntos
   - NO menciones visibilidad, resonancia u otros scores individuales

2. **Métricas**: Cambios porcentuales en publicaciones, interacciones y alcance
   - Usa los porcentajes EXACTOS de cambios_porcentuales
   - Describe si aumentaron, crecieron, cayeron, se mantuvieron estables

3. **Contenido destacado**: Análisis de las publicaciones
   - Identifica 2-3 temas principales del contenido
   - Menciona emociones, narrativas o eventos destacados
   - Explica cómo conectaron con la audiencia

4. **Cierre**: Número de comentarios del mes actual
   - Usa current_month.POST_COMMENTS__SUM
   - Formato: "Se registraron [número] comentarios/menciones durante el periodo"

**RESTRICCIONES CRÍTICAS:**
- LONGITUD: 95-100 palabras (cuenta cada palabra)
- NO incluir números de scores (105 puntos, 111 puntos, etc.)
- NO mencionar categorías individuales de scores a menos que sea excepcional
- UN SOLO PÁRRAFO, sin saltos de línea
- Español natural y profesional

**EJEMPLO CORRECTO (98 palabras):**
"En septiembre 2025, el Campus Monterrey mostró un desempeño satisfactorio, incrementando 86% su volumen de publicaciones, 152% las interacciones y 82% el alcance respecto al año anterior. Destacaron contenidos que combinaron nostalgia institucional y vida estudiantil auténtica: recorridos históricos del campus desde 1943, celebraciones patrias que reforzaron el orgullo mexicano, y momentos cotidianos como coffee breaks y el vibrante apoyo a Borregos. Estas narrativas generaron una conexión emocional que fortaleció el sentido de pertenencia y comunidad. Se registraron 556 comentarios durante el periodo, reflejando un engagement activo con la audiencia."

**EJEMPLO INCORRECTO:**
"En septiembre 2025, el Campus Monterrey mostró un desempeño satisfactorio con una salud de marca global de 106 puntos, impulsado por un crecimiento..." [❌ Incluye números de puntos]
"La resonancia alcanzó niveles satisfactorios (111 puntos) y la visibilidad fue también satisfactoria (105 puntos)..." [❌ Menciona scores individuales con números]
Párrafo de 150 palabras [❌ Excede límite de palabras]""",
    expected_output="CampusInsight con campus_id, campus_name y el insight completo en español de 95-100 palabras",
    output_pydantic=CampusInsight,
    agent=insight_agent,
)

# ============================================================================
# CREATE CREW
# ============================================================================
crew = Crew(
    agents=[insight_agent],
    tasks=[insight_task],
    process=Process.sequential,
    verbose=False,
)

# ============================================================================
# PROCESS EACH CAMPUS
# ============================================================================
print("🚀 Generating insights...\n")

insights = []
errors = []

for i, campus in enumerate(campuses, 1):
    campus_id = campus.get('campus_id', 'UNKNOWN')
    campus_name = campus.get('campus_name', 'Unknown')
    
    print(f"[{i}/{len(campuses)}] Processing {campus_id} - {campus_name}...", end=' ')
    
    try:
        campus_json = json.dumps(campus, ensure_ascii=False, indent=2)
        result = crew.kickoff(inputs={'campus_data': campus_json})
        
        if hasattr(result, 'pydantic'):
            insight_data = result.pydantic.dict()
            insights.append(insight_data)
            print(f"✅")
        else:
            print("❌ No pydantic output")
            errors.append(campus_id)
            
    except Exception as e:
        print(f"❌ Error: {str(e)[:60]}")
        errors.append(campus_id)

# Save insights JSON
output = {
    'insights': insights,
    'metadata': {
        'month': 'Septiembre 2025',
        'total_campuses': len(insights),
        'generated_at': datetime.now().strftime('%Y-%m-%d')
    }
}

with open('campus_insights.json', 'w', encoding='utf-8') as f:
    json.dump(output, f, ensure_ascii=False, indent=2)

print(f"\n{'='*70}")
print(f"✅ Generated: {len(insights)}/{len(campuses)} insights")
if errors:
    print(f"❌ Errors: {', '.join(errors)}")
print(f"💾 Saved: campus_insights.json")

# ============================================================================
# GENERATE MARKDOWN REPORT
# ============================================================================

if len(insights) > 0:
    print(f"{'='*70}\n")
    print("📝 Generating Markdown report...\n")
    
    # Sort alphabetically by campus name
    sorted_insights = sorted(insights, key=lambda x: x['campus_name'])
    
    # Build markdown content
    markdown_lines = [
        "# 📊 Reporte de Insights - Septiembre 2025\n",
        "## Resumen Ejecutivo\n",
        f"Análisis de desempeño en redes sociales de {len(insights)} campus durante septiembre 2025.\n",
        "\n---\n",
        "\n## Insights por Campus\n\n"
    ]
    
    # Add each campus
    for insight in sorted_insights:
        campus_name = insight['campus_name']
        campus_id = insight['campus_id']
        insight_text = insight['insight']
        
        markdown_lines.append(f"### 🎓 {campus_name} ({campus_id})\n\n")
        markdown_lines.append(f"{insight_text}\n\n")
        markdown_lines.append("---\n\n")
    
    # Add statistics
    markdown_lines.extend([
        "## Estadísticas Generales\n\n",
        f"- **Total de campus analizados:** {len(insights)}\n",
        "- **Periodo:** Septiembre 2025\n",
        f"- **Fecha de generación:** {datetime.now().strftime('%Y-%m-%d')}\n"
    ])
    
    # Save to file
    markdown_content = "".join(markdown_lines)
    with open('campus_insights_report.md', 'w', encoding='utf-8') as f:
        f.write(markdown_content)
    
    print("✅ Markdown report generated!")
    print("💾 Saved: campus_insights_report.md")
else:
    print("⚠️  No insights to format into Markdown")

print(f"{'='*70}")



📖 Loading unified campus data...
✅ Loaded 20 campuses

🚀 Generating insights...

[1/20] Processing MTY - Monterrey... ✅
[2/20] Processing CCM - Ciudad de México... ✅
[3/20] Processing LEO - León... ✅
[4/20] Processing SON - Sonora... ✅
[5/20] Processing CVA - Cuernavaca... ✅
[6/20] Processing PUE - Puebla... ✅
[7/20] Processing CEM - Estado de México... ✅
[8/20] Processing SIN - Sinaloa... ✅
[9/20] Processing SLP - San Luis Potosí... ✅
[10/20] Processing CSF - Santa Fe... ✅
[11/20] Processing AGS - Aguascalientes... ✅
[12/20] Processing GDL - Guadalajara... ✅
[13/20] Processing CHI - Chihuahua... ✅
[14/20] Processing QRO - Querétaro... ✅
[15/20] Processing CDJ - Cd. Juárez... ✅
[16/20] Processing COB - Cd. Obregón... ✅
[17/20] Processing TOL - Toluca... ✅
[18/20] Processing LAG - Laguna... ✅
[19/20] Processing SAL - Saltillo... ✅
[20/20] Processing HGO - Hidalgo... ✅

✅ Generated: 20/20 insights
💾 Saved: campus_insights.json

📝 Generating Markdown report...

✅ Markdown report generated